# NB02 - Create Embeddings

## Import dependencies

In [1]:
# General libraries
import sys
import os
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()

# Hugging Face Transformers
from transformers import AutoTokenizer, AutoModel

# Scikit-learn metrics
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Add the project root to Python path (assuming current working directory is 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Own functions
from utils import get_mean_embeddings, populate_doc_chunks_from_dict, populate_doc_chunks_from_json, count_chunks_for_doc, bulk_process_and_store_chunks, get_embeddings_from_json
from climate_policy_extractor.models import DocChunk, get_db_session
from sqlalchemy import create_engine, text, func

db_url = os.getenv('DATABASE_URL')

Download/Load embedding models

In [2]:
# Load ClimateBERT tokenizer and model
model_name = 'climatebert/distilroberta-base-climate-f'
local_model_save = '../local_models/climatebert/distilroberta-base-climate-f'

# # Uncomment to download model
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=False)
# model = AutoModelForMaskedLM.from_pretrained(model_name, use_auth_token=False)

# # Save it to a local_models folder
# tokenizer.save_pretrained(local_model_save)
# model.save_pretrained(local_model_save)

# Load the models from disk
tokenizer = AutoTokenizer.from_pretrained(local_model_save)
model = AutoModel.from_pretrained(local_model_save)

Some weights of the model checkpoint at ../local_models/climatebert/distilroberta-base-climate-f were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at ../local_models/climatebert/distilroberta-base-climate-f and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able t

In [3]:
# Run this if you want to load all chunks
all_chunks = json.load(open('../data/json/all_chunks_bert.json', 'r'))
all_chunks.keys()

# # Run this if you want to load a few chunks for testing
# all_chunks = json.load(open('../data/json/snippet_doc_chunks.json', 'r'))
# all_chunks.keys()

dict_keys(['afghanistan_english_20220601', 'albania_english_20220801', 'algeria_english_20220601', 'andorra_spanish_20230101', 'andorra_spanish_20250201', 'angola_english_20220601', 'antigua_and_barbuda_english_20220601', 'argentina_spanish_20220501', 'armenia_english_20220601', 'australia_english_20220601', 'azerbaijan_english_20231001', 'bahamas_english_20221101', 'bahrain_english_20220601', 'bangladesh_english_20220601', 'barbados_english_20220601', 'belarus_english_20220601', 'belize_english_20220601', 'benin_french_20220601', 'bhutan_english_20220601', 'bolivia_(plurinational_state_of)_english_20220601', 'bosnia_and_herzegovina_english_20220601', 'botswana_english_20220601', 'botswana_english_20241201', 'brazil_english_20241101', 'brunei_darussalam_english_20220601', 'burkina_faso_french_20220601', 'burundi_french_20220601', 'cabo_verde_english_20220601', 'cambodia_english_20220601', 'cameroon_french_20220601', 'canada_english_20220601', 'canada_english_20250201', 'central_african

In [5]:
# For multiple files
all_chunks_list = []

for doc, chunks_list in all_chunks.items():
    for chunk in chunks_list:
        # Add the filename as document ID to each chunk
        chunk = chunk.copy()  # Create a copy to avoid modifying the original
        chunk['doc_id'] = doc
        all_chunks_list.append(chunk)

# Create DataFrame from all chunks
all_chunks_df = pd.DataFrame(all_chunks_list)

# Get dims of DataFrame
all_chunks_df.head(2)

,id,type,text,page_number,metadata,doc_id
0,0,CompositeElement,ISLAMIC REPUBLIC OF AFGHANISTAN\n\nIntended Na...,1,"{'file_directory': '..\data\pdfs', 'filename':...",afghanistan_english_20220601
1,1,CompositeElement,Executive Summary\n\nBase Year:\n\n2005\n\nTar...,1,"{'file_directory': '..\data\pdfs', 'filename':...",afghanistan_english_20220601


WILL TAKE A LONG TIME FOR ALL CHUNKS

In [ ]:
all_chunks_df.loc[:, 'embedding'] = all_chunks_df['text'].progress_apply(lambda x: get_mean_embeddings(x, model, tokenizer))
all_chunks_df.to_json('../data/json/all_chunks_bert_embeddings.json', orient='records')
all_chunks_df.tail()

  0%|          | 0/65306 [00:00<?, ?it/s]

,id,type,text,page_number,metadata,doc_id,embedding
65301,302,CompositeElement,ow emissions development strategies and actions.,40,"{'file_directory': '..\data\pdfs', 'filename':...",zimbabwe_english_20250201,"[-0.0149321798235178, -0.018060462549328804, -..."
65302,303,CompositeElement,"5.2 CAPACITY\n\nBUILDING,\n\nEDUCATION,\n\nTRA...",40,"{'file_directory': '..\data\pdfs', 'filename':...",zimbabwe_english_20250201,"[0.02068692073225975, -0.09388821572065353, -0..."
65303,304,CompositeElement,The cross-cutting nature of the climate action...,40,"{'file_directory': '..\data\pdfs', 'filename':...",zimbabwe_english_20250201,"[0.017246006056666374, 0.017260683700442314, 0..."
65304,305,CompositeElement,the gender disparities,40,"{'file_directory': '..\data\pdfs', 'filename':...",zimbabwe_english_20250201,"[0.03902190178632736, 0.10949282348155975, -0...."
65305,306,CompositeElement,in\n\nthis NDC\n\ninformation and actions.\n\n...,40,"{'file_directory': '..\data\pdfs', 'filename':...",zimbabwe_english_20250201,"[0.001697609550319612, -0.01692597195506096, -..."


### Update Database with chunks and embeddings

In [ ]:
# # Load the JSON file into a DataFrame (run this if you already have the json with embeddings)
# with open('../data/json/all_chunks_bert_embeddings.json', 'r') as f:
#     all_chunks_dict = json.load(f)

# Call populate table function (run this if you are running the notebook for the first time)
all_chunks_dict = all_chunks_df.to_dict(orient='records')

all_chunks_dict[0]

{'id': 0,
 'type': 'CompositeElement',
 'text': 'ISLAMIC REPUBLIC OF AFGHANISTAN\n\nIntended Nationally Determined Contribution\n\nSubmission to the United Nations Framework Convention on Climate Change\n\n21 September 2015\n\n*****\n\nThe Islamic Republic of Afghanistan hereby communicates its Intended Nationally Determined Contribution (INDC) and information to facilitate understanding of the contribution.',
 'page_number': 1,
 'metadata': {'file_directory': '..\\data\\pdfs',
  'filename': 'afghanistan_english_20220601.pdf',
  'languages': ['eng'],
  'last_modified': '2025-03-24T10:38:04',
  'filetype': 'application/pdf'},
 'doc_id': 'afghanistan_english_20220601',
 'embedding': [0.0687589496,
  0.0500213914,
  -0.0266364403,
  0.0401619188,
  0.5135895014,
  0.1837328225,
  0.0577176176,
  0.0770772249,
  0.1838382483,
  -0.0776428133,
  -0.0614188574,
  0.357621938,
  0.000802157,
  -0.1518113464,
  0.0904157832,
  -0.2905346453,
  -0.1204291806,
  0.0422070585,
  0.1236632243,
  0

In [4]:
len(all_chunks_dict)

65306

In [5]:
# Connect to the database
db_session = get_db_session(db_url)

try:
    # Insert all chunks at once
    inserted = populate_doc_chunks_from_dict(
        chunks_list=all_chunks_dict,
        db_session=db_session,
        delete_existing=True,
        batch_size=1000
    )
    print(f"Inserted {inserted} chunks into the database")
finally:
    db_session.close()

Inserted 65306 chunks into the database


### Create vector similarity index

In [8]:
engine = create_engine(db_url)

with engine.connect() as conn:
        # Create the pgvector extension if it doesn't exist
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
        
        # Convert column type from ARRAY(Float) to vector
        # Only needed once after initial data load
        conn.execute(text("""
            ALTER TABLE doc_chunks 
            ALTER COLUMN embedding TYPE vector(768)
            USING embedding::vector(768);
        """))
        # Create an index on the embedding column for faster similarity search
        # The ivfflat index type is good for cosine similarity searches
        conn.execute(text("""
            CREATE INDEX IF NOT EXISTS chunks_embedding_idx 
            ON doc_chunks 
            USING ivfflat (embedding vector_cosine_ops)
            WITH (lists = 100);
        """))
        conn.commit()

Check the number of chunks corresponding to each doc_id in the database.

In [9]:
doc_id = "albania_english_20220801"
db_session = get_db_session(db_url)

try:
    count = count_chunks_for_doc(doc_id, db_session)
    print(f"Document {doc_id} is related to {count} rows in the doc_chunks table.")
finally:
    db_session.close()

Document albania_english_20220801 is related to 1071 rows in the doc_chunks table.
